In [1]:
import tarfile
import json
import os
import math

import pandas as pd
import numpy as np
import plotly.graph_objects as go

from pathlib import Path
from typing import List, Set
from plotly.subplots import make_subplots
from PIL import Image

In [2]:
def extract_uids_and_images_from_tar_files(tar_files_path: str | Path) -> pd.DataFrame:
    """
    Extract all UIDs and images from a set of tar files.
    
    Args:
        tar_files_path (str | Path): Path to directory containing tar files
    
    Returns:
        pd.DataFrame: DataFrame with UIDs and PIL image objects
    """
    import io
    from PIL import Image
    
    tar_files_path = Path(tar_files_path)
    uids_and_images = []
    
    # Get all tar files in the directory
    tar_files = [f for f in tar_files_path.glob("*.tar")]
    
    for tar_path in tar_files:
        try:
            with tarfile.open(tar_path, 'r') as tar:
                # Get all member names
                member_names = tar.getnames()
                
                # Filter to get just the JSON files
                json_files = [name for name in member_names if name.endswith('.json')]
                
                for json_file in json_files:
                    # Extract and read the JSON file to get the UID
                    json_member = tar.getmember(json_file)
                    json_fileobj = tar.extractfile(json_member)
                    
                    if json_fileobj is not None:
                        try:
                            metadata = json.load(json_fileobj)
                            uid = metadata.get("uid")
                            
                            if uid:
                                # Find the corresponding image file
                                image_name = json_file.replace('.json', '.jpg')
                                if image_name in member_names:
                                    image_member = tar.getmember(image_name)
                                    image_fileobj = tar.extractfile(image_member)
                                    
                                    if image_fileobj is not None:
                                        # Load the image into a PIL Image object
                                        image_data = image_fileobj.read()
                                        image = Image.open(io.BytesIO(image_data))
                                        
                                        # Add UID and image to the list
                                        uids_and_images.append({
                                            'uid': uid,
                                            'image': image
                                        })
                        except json.JSONDecodeError:
                            print(f"Error decoding JSON file {json_file} in {tar_path}")
        except tarfile.TarError as e:
            print(f"Error reading {tar_path}: {e}")
    
    # Create DataFrame from the list of dictionaries
    return pd.DataFrame(uids_and_images)


def filter_dataframe_by_uids(df: pd.DataFrame, filter_uids: List[str]) -> pd.DataFrame:
    """
    Filter DataFrame based on a given list of UIDs.
    
    Args:
        df (pd.DataFrame): DataFrame with UIDs and images
        filter_uids (List[str]): UIDs to filter by
    
    Returns:
        pd.DataFrame: Filtered DataFrame
    """
    filter_set = set(filter_uids)
    return df[df['uid'].isin(filter_set)]

# Example path to tar files
tar_directory = "/home/fbernardi/Documents/fair_spoke_8/data_quality_pipeline/test/data"

# Extract all UIDs and images
df_images = extract_uids_and_images_from_tar_files(tar_directory)
print(f"Total images found: {len(df_images)}")


Total images found: 4070


In [3]:
semdedup_metadata_path = '/home/fbernardi/Documents/fair_spoke_8/data_quality_pipeline/src/made/models/semdedup/data'

# Distances to centroids
dist_to_cent = np.load(
    os.path.join(
        semdedup_metadata_path,
        'clustering', 
        'dist_to_cent.npy'
    )
)

# Vectors of centroids
centroids = np.load(
    os.path.join(
        semdedup_metadata_path,
        'clustering', 
        'kmeans_centroids.npy'
    )
)

nearest_cent = np.load(
    os.path.join(
        semdedup_metadata_path,
        'clustering', 
        'nearest_cent.npy'
    )
)

In [23]:
import random

tot_clusters = len(os.listdir(
        os.path.join(
            semdedup_metadata_path, 'sorted_clusters')
    ))

cluster_id = random.randint(0, tot_clusters - 1)
eps = 0.1

pruning_table = pd.read_pickle(
    os.path.join(
        semdedup_metadata_path,
        'dataframes',
        f'cluster_{cluster_id}.pkl'
    )
)

cluster = np.load(
    os.path.join(
        semdedup_metadata_path,
        'sorted_clusters', 
        f'cluster_{cluster_id}.npy'
    )
)

true_uids  = pruning_table[pruning_table[f"eps={eps}"]==True]["cluster_uids"].to_list()
false_uids  = pruning_table[pruning_table[f"eps={eps}"]==False]["cluster_uids"].to_list()

print(
    f"\nCluster ID: {cluster_id}",
    f"\nTrue UIDs: {len(true_uids)}",
    f"\nFalse UIDs: {len(false_uids)}",
    f"\nTotal UIDs: {len(true_uids) + len(false_uids)}",
)


Cluster ID: 73 
True UIDs: 0 
False UIDs: 34 
Total UIDs: 34


In [ ]:
def plot_images_matrix_plotly_chunked(df_filtered, chunk_size=50, max_cols=6, 
                                     width_per_image=200, height_per_image=200, 
                                     show_progress=True):
    """
    Plot PIL images with UIDs as titles using Plotly, handling large datasets by chunking.
    
    Args:
        df_filtered: DataFrame with 'image' and 'uid' columns
        chunk_size: Number of images per plot/chunk
        max_cols: Maximum number of columns in each grid
        width_per_image: Width of each image in pixels
        height_per_image: Height of each image in pixels
        show_progress: Whether to print progress information
    """
    if (chunk_size is None) or (chunk_size < 50):
        chunk_size = 50
    
    total_images = len(df_filtered)
    if total_images == 0:
        print("No images to display")
        return
        
    # Calculate number of chunks needed
    num_chunks = math.ceil(total_images / chunk_size)
    
    if show_progress:
        print(f"Total images: {total_images}")
        print(f"Chunk size: {chunk_size}")
        print(f"Number of plots to generate: {num_chunks}")
        print("-" * 50)
    
    # Process each chunk
    for chunk_idx in range(num_chunks):
        start_idx = chunk_idx * chunk_size
        end_idx = min(start_idx + chunk_size, total_images)
        
        # Get current chunk of data
        chunk_df = df_filtered.iloc[start_idx:end_idx].copy()
        chunk_size_actual = len(chunk_df)
        
        if show_progress:
            print(f"Processing chunk {chunk_idx + 1}/{num_chunks}: "
                  f"images {start_idx + 1}-{end_idx} ({chunk_size_actual} images)")
        
        # Calculate grid dimensions for this chunk
        n_cols = min(max_cols, chunk_size_actual)
        n_rows = math.ceil(chunk_size_actual / n_cols)
        
        # Create subplot titles with UIDs
        subplot_titles = []
        for _, row in chunk_df.iterrows():
            uid_short = row['uid'][:8] + '...' if len(row['uid']) > 8 else row['uid']
            subplot_titles.append(uid_short)
        
        # Fill remaining titles with empty strings
        subplot_titles.extend([''] * (n_rows * n_cols - chunk_size_actual))
        
        # Create subplots for this chunk
        fig = make_subplots(
            rows=n_rows, 
            cols=n_cols,
            subplot_titles=subplot_titles,
            vertical_spacing=0.08,
            horizontal_spacing=0.05
        )
        
        # Add images to subplots
        for i, (_, row) in enumerate(chunk_df.iterrows()):
            plot_row = (i // n_cols) + 1  # Plotly uses 1-indexed
            plot_col = (i % n_cols) + 1
            
            # Convert PIL image to numpy array
            # Get the image and resize if it's too small
            img = row['image']
            min_size = 300  # Minimum size in pixels
            if img.width < min_size or img.height < min_size:
                # Maintain aspect ratio while ensuring both dimensions are at least min_size
                ratio = max(min_size / img.width, min_size / img.height)
                new_size = (int(img.width * ratio), int(img.height * ratio))
                img = img.resize(new_size, resample=Image.LANCZOS if hasattr(Image, 'LANCZOS') else Image.BICUBIC)
            img_array = np.array(img)
            
            # Add image to subplot
            fig.add_trace(
                go.Image(z=img_array),
                row=plot_row, col=plot_col
            )
        
        # Update layout
        chunk_title = f"Image Matrix - Chunk {chunk_idx + 1}/{num_chunks} (Images {start_idx + 1}-{end_idx})"
        fig.update_layout(
            title=chunk_title,
            showlegend=False,
            width=n_cols * width_per_image,
            height=n_rows * height_per_image + 150  # Extra space for titles
        )
        
        # Remove axes for all subplots
        for i in range(1, n_rows + 1):
            for j in range(1, n_cols + 1):
                fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False, row=i, col=j)
                fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False, row=i, col=j)
        
        # Show the plot for this chunk
        fig.show()
        
        if show_progress and chunk_idx < num_chunks - 1:
            print(f"Chunk {chunk_idx + 1} completed. Preparing next chunk...")
            print()

def plot_images_from_filter(df_images, uid_filter, chunk_size=50, max_cols=6, 
                           width_per_image=200, height_per_image=200):
    """
    Convenience function to filter and plot images in chunks.
    
    Args:
        df_images: DataFrame with 'image' and 'uid' columns
        uid_filter: List/Series of UIDs to filter by
        chunk_size: Number of images per plot/chunk
        max_cols: Maximum number of columns in each grid
        width_per_image: Width of each image in pixels
        height_per_image: Height of each image in pixels
    """
    # Filter the dataframe
    filtered_df = df_images[df_images['uid'].isin(uid_filter)]
    
    print(f"Filtered {len(filtered_df)} images from {len(df_images)} total images")
    print(f"Filter contains {len(uid_filter)} unique UIDs")
    print()
    
    # Plot in chunks
    plot_images_matrix_plotly_chunked(
        filtered_df, 
        chunk_size=chunk_size, 
        max_cols=max_cols,
        width_per_image=width_per_image, 
        height_per_image=height_per_image
    )

In [ ]:
def plot_comparison_chunks(df_images, true_uids, false_uids, chunk_size=50, max_cols=6):
    """
    Plot both true and false images in chunks for comparison.
    
    Args:
        df_images: DataFrame with image data
        true_uids: UIDs for true/positive images
        false_uids: UIDs for false/negative images
        chunk_size: Number of images per chunk
        max_cols: Maximum columns per grid
    """
    print("=== PLOTTING TRUE IMAGES ===")
    plot_images_from_filter(df_images, true_uids, chunk_size, max_cols)
    
    print("\n" + "="*50)
    print("=== PLOTTING FALSE IMAGES ===")
    plot_images_from_filter(df_images, false_uids, chunk_size, max_cols)

def get_chunk_info(total_images, chunk_size, max_cols=6):
    """
    Calculate and display information about how images will be chunked.
    
    Args:
        total_images: Total number of images
        chunk_size: Images per chunk
        max_cols: Maximum columns per grid
    """
    num_chunks = math.ceil(total_images / chunk_size)
    
    print(f"Chunking Information:")
    print(f"- Total images: {total_images}")
    print(f"- Chunk size: {chunk_size}")
    print(f"- Number of chunks: {num_chunks}")
    print(f"- Max columns per grid: {max_cols}")
    
    for i in range(num_chunks):
        start_idx = i * chunk_size
        end_idx = min(start_idx + chunk_size, total_images)
        images_in_chunk = end_idx - start_idx
        rows_needed = math.ceil(images_in_chunk / max_cols)
        
        print(f"  Chunk {i+1}: {images_in_chunk} images, {rows_needed} rows x {min(max_cols, images_in_chunk)} cols")

# Plots

# Per source image


In [79]:
tot_clusters = len(os.listdir(
        os.path.join(
            semdedup_metadata_path, 'sorted_clusters')
    ))

cluster_id = random.randint(0, tot_clusters - 1)

pruning_table = pd.read_pickle(
    os.path.join(
        semdedup_metadata_path,
        'dataframes',
        f'cluster_{cluster_id}.pkl'
    )
)

sorted_cluster = np.load(
    os.path.join(
        semdedup_metadata_path,
        'sorted_clusters', 
        f'cluster_{cluster_id}.npy'
    )
)

uid_pruning = pruning_table.loc[0, 'cluster_uids']
image_id_in_dataset = pruning_table.loc[0, 'image_id_in_dataset']
image_id_in_dataset = int(image_id_in_dataset)
uid_dataset = df_images.loc[image_id_in_dataset, 'uid']

print(pruning_table.image_id_in_dataset.apply(type).value_counts())

assert uid_dataset == uid_pruning

print(
    f"\n ID: {image_id_in_dataset}",
    f"\n UID: {uid_dataset}",
    f"\n Total UIDs: {len(pruning_table)}",
    f"\n Nearest centroid: {cluster_id}",
    f"\n Distance: {1- dist_to_cent[image_id_in_dataset]}",
    f"\n UIDs match: {uid_dataset == uid_pruning}",
    f"\n Is the closest image to center?: {
        pruning_table.iloc[:, 3:].sum(axis=1).argmin() == 
        pruning_table.loc[pruning_table['image_id_in_dataset'].astype(int) == image_id_in_dataset].index[0]
    }"
)

image = df_images.loc[image_id_in_dataset, 'image']
# Display the PIL image using plotly
import plotly.express as px
img_array = np.array(image)
fig = px.imshow(img_array)
fig.update_layout(coloraxis_showscale=False)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.show()

# Basic plot
fig1 = plot_cluster_images_plotly(
    pruning_table=pruning_table,
    df_images=df_images,
    list_of_eps=list_of_eps,
    cluster_id=cluster_id,
    closest_uid=uid_dataset,
    width=2000,
    height=400,
    max_images=8
)

fig1.show()

image_id_in_dataset
<class 'int'>    37
Name: count, dtype: int64

 ID: 20 
 UID: 8d46ea342e070ed6e763ec355f0f525e 
 Total UIDs: 37 
 Nearest centroid: 92 
 Distance: 0.15254545211791992 
 UIDs match: True 
 Is the closest image to center?: True


In [26]:
# Original list of epsilon values
list_of_eps = [
    0.00001, 0.00002, 0.00005, 0.0001, 0.0002, 0.0005, 0.001, 
    0.002, 0.005, 0.01, 0.02, 0.03, 0.04, 0.05, 
    0.06, 0.07, 0.08, 0.09, 0.1, 0.11, 0.12, 
    0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19,
    0.2, 0.21, 0.22, 0.23, 0.24, 0.25, 0.26, 
    0.27, 0.28, 0.29, 0.3, 0.32, 0.34, 0.36, 
    0.38, 0.4, 0.42, 0.44, 0.46, 0.48, 0.5,
    0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85,
    0.9, 0.95, 1.0, 1.1, 1.2, 1.3, 1.4, 
    1.5, 1.6, 1.7, 1.8, 1.9, 2.0, 2.2, 2.4,
    2.6, 2.8, 3.0, 3.2, 3.4, 3.6, 3.8, 
    4.0, 4.2, 4.4, 4.6, 4.8, 5.0
]

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import numpy as np

def plot_cluster_images_plotly(pruning_table, df_images, list_of_eps, cluster_id, closest_uid,
                              width=1200, height=400, max_images=10):
    """
    Plot images from a cluster in a row using Plotly, starting with the closest to centroid,
    followed by the first image that is True in each epsilon column.
    
    Parameters:
    -----------
    pruning_table : pd.DataFrame
        The pruning table for the cluster with columns:
        - 'cluster_uids': UIDs of images
        - 'image_id_in_dataset': Dataset indices
        - 'eps=<value>': Boolean columns for each epsilon threshold
    df_images : pd.DataFrame
        DataFrame with images, must have 'image' column accessible by image_id_in_dataset
    list_of_eps : list
        List of epsilon values to check
    cluster_id : int
        ID of the cluster being plotted
    closest_uid : str
        UID of the image closest to centroid
    width : int
        Figure width in pixels
    height : int
        Figure height in pixels
    max_images : int
        Maximum number of images to plot
        
    Returns:
    --------
    plotly.graph_objects.Figure
    """
    
    images_to_plot = []
    titles = ['Closest to\ncentroid']
    
    # 1. First image: closest to centroid (provided as parameter)
    closest_idx = pruning_table[pruning_table['cluster_uids'] == closest_uid].index[0]
    closest_image_id = pruning_table.loc[closest_idx, 'image_id_in_dataset']
    
    images_to_plot.append(df_images.loc[closest_image_id, 'image'])
    
    true_indices_old = []
    # 2. For each epsilon, find first image that is True (marked for removal)
    for eps in list_of_eps:
        eps_col = f'eps={eps}'
        
        if eps_col in pruning_table.columns:
            # Find first True value in this epsilon column
            true_indices = pruning_table[pruning_table[eps_col] == True].index
            
            if (list(true_indices) != true_indices_old):
                for idx in true_indices:
                    image_id = pruning_table.loc[idx, 'image_id_in_dataset']
                    image = df_images.loc[image_id, 'image']
                    
                    if image not in images_to_plot:
                        images_to_plot.append(image)
                        titles.append(f'eps={eps}')
            
                true_indices_old = list(true_indices)
    
        # Stop if we've reached max_images
        if len(images_to_plot) >= max_images:
            break
    
    # Create subplots
    n_images = len(images_to_plot)
    if n_images == 0:
        print("No images to plot")
        return None
    
    fig = make_subplots(
        rows=1, cols=n_images,
        subplot_titles=titles,
        horizontal_spacing=0.02
    )
    
    # Add images to subplots
    for i, image in enumerate(images_to_plot):
        # Convert PIL Image to numpy array if needed
        if hasattr(image, 'convert'):
            img_array = np.array(image.convert('RGB'))
        else:
            img_array = np.array(image)
        
        fig.add_trace(
            go.Image(z=img_array),
            row=1, col=i+1
        )
    
    # Update layout
    fig.update_layout(
        title=f'Cluster {cluster_id} - Semantic Deduplication Analysis',
        width=width,
        height=height,
        showlegend=False
    )
    
    # Remove axes for all subplots
    for i in range(1, n_images + 1):
        fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False, row=1, col=i)
        fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False, row=1, col=i)
    
    return fig

In [ ]:
# Alternative version using px.imshow for each image (sometimes renders better)
def plot_cluster_images_plotly_alt(pruning_table, df_images, list_of_eps, cluster_id, closest_uid,
                                  width=1200, height=400, max_images=10):
    """
    Alternative Plotly implementation using px.imshow for potentially better rendering.
    
    Parameters are the same as plot_cluster_images_plotly.
    
    Returns:
    --------
    plotly.graph_objects.Figure
    """
    
    images_to_plot = []
    titles = []
    
    # 1. First image: closest to centroid
    closest_idx = pruning_table[pruning_table['cluster_uids'] == closest_uid].index[0]
    closest_image_id = pruning_table.loc[closest_idx, 'image_id_in_dataset']
    
    images_to_plot.append(df_images.loc[closest_image_id, 'image'])
    titles.append('Closest to<br>centroid')
    
    true_indices_old = []
    # 2. For each epsilon, find first image that is True (marked for removal)
    for eps in list_of_eps:
        eps_col = f'eps={eps}'
        
        if eps_col in pruning_table.columns:
            # Find first True value in this epsilon column
            true_indices = pruning_table[pruning_table[eps_col] == True].index
            
            if (list(true_indices) != true_indices_old):
                for idx in true_indices:
                    image_id = pruning_table.loc[idx, 'image_id_in_dataset']
                    image = df_images.loc[image_id, 'image']
                    
                    if image not in images_to_plot:
                        images_to_plot.append(image)
                        titles.append(f'First True<br>eps={eps}')
            
                true_indices_old = list(true_indices)
    
        # Stop if we've reached max_images
        if len(images_to_plot) >= max_images:
            break
    
    n_images = len(images_to_plot)
    if n_images == 0:
        print("No images to plot")
        return None
    
    # Create subplots
    fig = make_subplots(
        rows=1, cols=n_images,
        subplot_titles=titles,
        horizontal_spacing=0.02
    )
    
    # Add each image using px.imshow approach
    for i, image in enumerate(images_to_plot):
        # Convert PIL Image to numpy array if needed
        if hasattr(image, 'convert'):
            img_array = np.array(image.convert('RGB'))
        else:
            img_array = np.array(image)
        
        # Create individual imshow figure and extract its data
        temp_fig = px.imshow(img_array)
        img_trace = temp_fig.data[0]
        
        fig.add_trace(img_trace, row=1, col=i+1)
    
    # Update layout
    fig.update_layout(
        title=f'Cluster {cluster_id} - Semantic Deduplication Analysis',
        width=width,
        height=height,
        showlegend=False
    )
    
    # Remove axes
    for i in range(1, n_images + 1):
        fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False, row=1, col=i)
        fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False, row=1, col=i)
    
    return fig

## In cluster

In [ ]:
#  Rejected samples
plot_images_from_filter(df_images, true_uids, chunk_size=10, max_cols=6)

In [ ]:
# Kept samples
plot_images_from_filter(df_images, false_uids, chunk_size=60, max_cols=8)